# Climate Change Signal — DestinE Generation 2

Compare **30-year mean 2m temperature between historical and scenario experiments** across IFS-FESOM, IFS-NEMO, and ICON using monthly (`clmn`) data from the DestinE Climate DT Generation 2 simulations.

In [ ]:
from destine_climate_helpers import fetch_period
import healpy as hp
import matplotlib.pyplot as plt

In [ ]:
# --- Configuration ---
PARAM = 'avg_2t'
PARAM_NAME = '2m Temperature'
MODELS = ['IFS-NEMO','IFS-FESOM'] # ['IFS-NEMO','IFS-FESOM','ICON']
RESOLUTION = 'standard'
STORE_DATA = True          # Set True to cache NetCDF files to DATA_DIR
DATA_DIR = './data'

# Historical period
HIST_YEARS = range(1990, 2015)
HIST_EXPERIMENT = 'hist'        # activity: baseline

# Future period
SCEN_YEARS = range(2015, 2050)
SCEN_EXPERIMENT = 'SSP3-7.0'   # activity: projections

In [ ]:
# Download monthly data for all models and both periods
data = {}
for model in MODELS:
    print(f'\n=== {model} ===')
    
    print(f'Fetching {HIST_EXPERIMENT} ({HIST_YEARS[0]}-{HIST_YEARS[-1]})...')
    hist = fetch_period(model, HIST_EXPERIMENT, HIST_YEARS, PARAM, RESOLUTION, STORE_DATA, DATA_DIR)
    
    print(f'Fetching {SCEN_EXPERIMENT} ({SCEN_YEARS[0]}-{SCEN_YEARS[-1]})...')
    scen = fetch_period(model, SCEN_EXPERIMENT, SCEN_YEARS, PARAM, RESOLUTION, STORE_DATA, DATA_DIR)
    
    data[model] = {'hist': hist, 'scen': scen}

- **The resulting data is cached in the following folder structure** if you have set `STORE_DATA = True`

In [ ]:
!ls data/{ICON,IFS-FESOM,IFS-NEMO}/{hist,SSP*}/clmn/standard/*nc | head -20

- **Your data is stored in an `xarray.DataArray`**, grouped by model name and experiment

In [ ]:
data['IFS-NEMO']['hist'] # ICON/IFS-NEMO/IFS-FESOM + hist/scen

In [ ]:
# Compute climate change signal: future mean minus historical mean
diffs = {}
for model in MODELS:
    hist_mean = data[model]['hist'].mean(dim='valid_time')
    scen_mean = data[model]['scen'].mean(dim='valid_time')
    diffs[model] = scen_mean - hist_mean

In [ ]:
# Plot climate change signal for each model
vmin = -2.5
vmax = 2.5

fig = plt.figure(figsize=(18, 5))
for i, model in enumerate(MODELS):
    hp.mollview(
        diffs[model].values, nest=True, flip='geo',
        cmap='RdBu_r', min=vmin, max=vmax,
        title=f'{model}: \u0394T ({SCEN_EXPERIMENT} {SCEN_YEARS[0]}-{SCEN_YEARS[-1]}'
              f' minus {HIST_EXPERIMENT} {HIST_YEARS[0]}-{HIST_YEARS[-1]})',
        unit='K', sub=(1, 3, i + 1),
    )

In [ ]:
# Global mean temperature change per model
# (HEALPix pixels are equal-area, so a simple mean is area-weighted)
for model in MODELS:
    print(f'{model}: global mean \u0394T = {float(diffs[model].mean()):.2f} K')

## How to extend this notebook for your analysis

- **Change variable**: set `PARAM` to another code (e.g. `'235043'` for total precipitation), update `PARAM_NAME`
- **Seasonal breakdown**: after computing `diffs`, you can group by month using `xarray.groupby('valid_time.season')`
- **Store data locally**: set `STORE_DATA = True` and adjust `DATA_DIR`; subsequent runs will then load from cache